In [ ]:
#@title ⚙️ نصب و دانلود (pocket-tts-farsi-v2 + G2P فارسی)

HF_TOKEN = "" #@param {type:"string"}
USE_GPU = True #@param {type:"boolean"}

import os, sys, subprocess, shutil

if HF_TOKEN.strip():
    os.environ['HF_TOKEN'] = HF_TOKEN.strip()
    print('✅ توکن Hugging Face تنظیم شد.')
else:
    os.environ.pop('HF_TOKEN', None)
    print('ℹ️ بدون توکن Hugging Face ادامه می‌دهیم.')

print('=' * 40)

# ── ۱) PyTorch — با --index-url (نه --extra-index-url) تا نسخه‌ی CPU جایگزین نشود
if USE_GPU:
    print('⏳ نصب PyTorch (cu128) ...')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'torch', 'torchaudio',
                        '--index-url', 'https://download.pytorch.org/whl/cu128'])
    if r.returncode != 0:
        print('⚠️ ایندکس cu128 ناموفق — نصب نسخه‌ی پیش‌فرض PyPI.')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchaudio'], check=True)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchaudio'], check=True)

# ── ۲) pocket-tts از فورک مهدی الهیاری
# نسخه‌ی PyPI کلیدهای model.yaml این مدل را نمی‌شناسد و با
# ValidationError: capitalize_first_letter — Extra inputs are not permitted
# شکست می‌خورد. آن کلیدها عمداً خطا می‌دهند تا حرف اول کلمات بی‌صدا حذف نشود.
print('⏳ نصب pocket-tts (فورک فارسی) ...')
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'pocket-tts @ git+https://github.com/mallahyari/pocket-tts@main'])
if r.returncode != 0:
    print('⚠️ نصب فورک ناموفق بود؛ تلاش دوباره با --no-deps ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
                    'pocket-tts @ git+https://github.com/mallahyari/pocket-tts@main'], check=True)

# ── ۳) G2P و ابزارهای کمکی — بدون -U روی ipywidgets تا ویجت‌های Colab نشکنند
print('⏳ نصب transformers و ابزارها ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'sentencepiece', 'python-docx', 'pydub',
                'soundfile', 'scipy', 'ipywidgets'], check=True)

# ── ۴) ffmpeg
if shutil.which('ffmpeg') is None:
    print('⏳ نصب ffmpeg ...')
    subprocess.run('apt-get update -qq && apt-get install -y -qq ffmpeg', shell=True, check=True)

import torch
from pocket_tts import TTSModel

GPU_OK = torch.cuda.is_available() and USE_GPU
DEVICE = 'cuda' if GPU_OK else 'cpu'

if USE_GPU and not torch.cuda.is_available():
    print('⚠️ GPU در دسترس نیست — روی CPU اجرا می‌شود (این مدل روی CPU هم سریع است).')

if GPU_OK:
    props = torch.cuda.get_device_properties(0)
    print(f'🎮 GPU: {torch.cuda.get_device_name(0)} ({props.total_memory/1e9:.1f} GB)')
else:
    print('🖥️ اجرا روی CPU — pocket-tts برای CPU طراحی شده و ~۶ برابر سریع‌تر از زمان واقعی است.')

print('✅ نصب کامل شد. مدل‌ها در سلول بعدی لود می‌شوند.')


In [ ]:
#@title 🧠 لود مدل TTS و موتور G2P فارسی

import os, inspect, importlib, sys
import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration

# اگر سلول نصب اجرا نشده باشد، خطای واضح بدهد نه NameError
try:
    from pocket_tts import TTSModel
except ImportError as e:
    raise RuntimeError(
        '❌ pocket-tts نصب نیست. ابتدا سلول «⚙️ نصب و دانلود» را اجرا کنید.'
    ) from e

MODEL_REPO   = 'mehdi-hf/pocket-tts-farsi-v2'
MODEL_CONFIG = f'hf://{MODEL_REPO}/model.yaml'
G2P_REPO     = 'mehdi-hf/Homo-GE2PE-Persian-HF'

Temperature   = 0.3  #@param {type:"slider", min:0.1, max:1.0, step:0.05}
EosThreshold  = -4.0 #@param {type:"number"}

# ── نرمال‌ساز فارسی: در خودِ ریپوی مدل منتشر شده (normalize_fa.py)
print('⏳ دریافت نرمال‌ساز فارسی ...')
from huggingface_hub import hf_hub_download
_norm_path = hf_hub_download(MODEL_REPO, 'normalize_fa.py')
_norm_dir = os.path.dirname(_norm_path)
if _norm_dir not in sys.path:
    sys.path.insert(0, _norm_dir)
from normalize_fa import normalize_for_model

# ── مدل TTS
print('⏳ لود مدل pocket-tts-farsi-v2 (۱۰۹.۵M پارامتر، ~۴۴۰ مگابایت) ...')

_sig = inspect.signature(TTSModel.load_model)
_kw = {}
if 'config' in _sig.parameters:
    _kw['config'] = MODEL_CONFIG
if 'temp' in _sig.parameters:
    _kw['temp'] = float(Temperature)
if 'eos_threshold' in _sig.parameters:
    _kw['eos_threshold'] = float(EosThreshold)

try:
    tts_model = TTSModel.load_model(**_kw)
except TypeError as e:
    print(f'⚠️ امضای load_model متفاوت است ({e}) — تلاش با آرگومان موضعی.')
    tts_model = TTSModel.load_model(MODEL_CONFIG)

if GPU_OK:
    tts_model = tts_model.to(DEVICE)
    print(f'🎮 مدل روی {DEVICE} منتقل شد.')

SAMPLE_RATE = getattr(tts_model, 'sample_rate', 24000)

# ── موتور G2P (نوشتار فارسی → فونم)
print('⏳ لود G2P (Homo-GE2PE، ~۳۳ مگابایت) ...')
g2p_tok = AutoTokenizer.from_pretrained(G2P_REPO)
g2p_model = T5ForConditionalGeneration.from_pretrained(G2P_REPO).eval().to(DEVICE)

# نگاشت نمادهای داخلی GE2PE به نوشتار مدل TTS
_TO_PHONEMES = str.maketrans({'/': 'a', 'a': 'A', '@': '?', '$': 'S', 'c': 'C'})


def phonemise(text: str) -> str:
    """متن فارسی → فونم رومانیزه. خروجی خالی یعنی مدل چیزی نخواهد گفت."""
    if not text or not text.strip():
        return ''
    try:
        t = normalize_for_model(text)          # اعداد، ی/ك عربی، اعراب، نیم‌فاصله
    except ValueError:
        # نرمال‌ساز عمداً استثنا می‌اندازد وقتی متن کاملاً بیرون از الفبای فارسی است
        # (فقط علامت، فقط Latin، یا خالی). یک واحد خراب نباید کل سند را بکشد.
        return ''
    t = t.replace('؟', '').replace('?', '')    # ؟ با نماد انسدادی چاکنایی اشتباه می‌شود
    if not t.strip():
        return ''
    enc = g2p_tok([t], add_special_tokens=False, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = g2p_model.generate(**enc, num_beams=5, max_length=512, early_stopping=True)
    raw = g2p_tok.batch_decode(out, skip_special_tokens=True)[0].strip()
    # "1" پرچم داخلی اضافه‌ی (ezafe) است و خودش فونم نیست
    return raw.translate(_TO_PHONEMES).replace('1', '').strip()


print()
print('✅ همه چیز آماده است.')
print(f'   TTS: {MODEL_REPO}  |  SR: {SAMPLE_RATE} Hz  |  device: {DEVICE}')
print(f'   G2P: {G2P_REPO}')
print()
print('🔍 نمونه‌ی تبدیل:')
for _s in ['سلام، حال شما چطور است؟', 'تا سال ۲۰۳۰ تغییر دهد.', 'اقتصاد آمریکا را تغییر دهد']:
    print(f'   «{_s}»')
    print(f'   → {phonemise(_s)}')


In [ ]:
#@title 📂 انتخاب فایل متنی (گوگل درایو یا حافظه داخلی)

import os

Use_Local_File = True #@param {type:"boolean"}

TEXT_EXTENSIONS = ('.txt', '.docx', '.md', '.rtf', '.srt', '.vtt', '.sub', '.ass')

SELECTED_DOC_PATH = None

if Use_Local_File:

    from google.colab import files

    print('📁 لطفاً فایل متنی خود را انتخاب کنید...')
    uploaded = files.upload()

    if not uploaded:
        raise RuntimeError('❌ فایلی انتخاب نشد!')

    fname = list(uploaded.keys())[0]

    if not fname.lower().endswith(TEXT_EXTENSIONS):
        raise RuntimeError(
            '❌ این فرمت پشتیبانی نمی‌شود!\n'
            f'فرمت‌های مجاز: {", ".join(TEXT_EXTENSIONS)}'
        )

    SELECTED_DOC_PATH = os.path.abspath(fname)
    print(f'✅ فایل انتخاب شد: {SELECTED_DOC_PATH}')

else:

    from google.colab import drive
    import ipywidgets as widgets
    from IPython.display import display

    print('🔄 در حال اتصال به Google Drive...')
    drive.mount('/content/drive', force_remount=True)

    ROOT_DIR = '/content/drive/MyDrive'
    current_dir = [ROOT_DIR]

    path_label = widgets.HTML()
    listing = widgets.Select(rows=14, layout=widgets.Layout(width='100%'))
    btn_open = widgets.Button(description='📂 باز کردن / تأیید انتخاب', button_style='primary')
    btn_up = widgets.Button(description='⬆️ پوشه بالاتر')
    status = widgets.HTML()

    def refresh_listing():
        path_label.value = f'<b>📍 مسیر فعلی:</b> {current_dir[0]}'
        try:
            entries = sorted(os.listdir(current_dir[0]), key=lambda x: x.lower())
        except Exception as e:
            status.value = f'<span style="color:red">❌ خطا در خواندن پوشه: {e}</span>'
            listing.options = []
            return

        options = []
        for e in entries:
            full = os.path.join(current_dir[0], e)
            if os.path.isdir(full):
                options.append(f'📁 {e}')
            elif e.lower().endswith(TEXT_EXTENSIONS):
                options.append(f'📄 {e}')

        listing.options = options
        status.value = ('' if options else
                        '<span style="color:#888">📭 فایل متنی پشتیبانی‌شده‌ای در این پوشه نیست.</span>')

    def on_up_clicked(b):
        if os.path.normpath(current_dir[0]) == os.path.normpath(ROOT_DIR):
            status.value = '<span style="color:orange">⚠️ شما در پوشه اصلی Drive هستید.</span>'
            return
        parent = os.path.dirname(current_dir[0].rstrip('/'))
        if parent and len(parent) >= len(ROOT_DIR):
            current_dir[0] = parent
            refresh_listing()

    def on_open_clicked(b):
        global SELECTED_DOC_PATH

        if not listing.value:
            status.value = '<span style="color:orange">⚠️ ابتدا یک فایل انتخاب کنید.</span>'
            return

        name = listing.value[2:].strip()
        full = os.path.join(current_dir[0], name)

        if os.path.isdir(full):
            current_dir[0] = full
            refresh_listing()
            return

        if full.lower().endswith(TEXT_EXTENSIONS):
            SELECTED_DOC_PATH = full
            status.value = ('<div style="margin-top:8px;padding:10px;border-radius:6px;'
                            'background:#e8f5e9;color:#1b5e20;font-weight:bold;">'
                            f'✅ فایل تأیید شد.<br>📄 {name}</div>')
            print(f'✅ فایل انتخاب شد:\n{SELECTED_DOC_PATH}')

    btn_up.on_click(on_up_clicked)
    btn_open.on_click(on_open_clicked)

    refresh_listing()
    display(path_label, listing, widgets.HBox([btn_up, btn_open]), status)
    print('راهنما: پوشه‌ها را انتخاب کنید و «📂 باز کردن / تأیید انتخاب» را بزنید.')


In [ ]:
#@title 🎙️ آپلود صدای مرجع برای کلون

from google.colab import files
import os, json, shutil, subprocess

#@markdown **مهم‌ترین تنظیم این نوت‌بوک:** صدای مرجع باید **۵ ثانیه یا کمتر** باشد.
#@markdown مدل روی پرامپت‌های حداکثر ۵ ثانیه آموزش دیده؛ صدای بلندتر باعث می‌شود
#@markdown مدل به‌جای خواندن متن شما، جمله‌ی خودِ پرامپت را ادامه دهد.
#@markdown در اندازه‌گیری رسمی: پرامپت ۶.۱ ثانیه‌ای → ۰ از ۴ بار کلمه‌ی خواسته‌شده گفته نشد؛
#@markdown همان کلیپ بریده‌شده به ۵ ثانیه → ۴ از ۴ بار درست بود.
Voice_Prompt_Seconds = 5 #@param {type:"slider", min:3, max:8, step:1}
#@markdown ---
#@markdown #### 🗣️ تنظیمات تولید
Max_Phoneme_Tokens = 18 #@param {type:"slider", min:8, max:24, step:1}
Frames_After_Eos = 0 #@param {type:"integer"}
Pause_Between_Segments_ms = 250 #@param {type:"slider", min:0, max:800, step:50}

if Voice_Prompt_Seconds > 5:
    print('⚠️ بالای ۵ ثانیه خارج از توزیع آموزش است — ممکن است متن شما خوانده نشود.')

if Max_Phoneme_Tokens > 20:
    print('⚠️ بالای ۲۰ توکن، تولید ممکن است هرگز تمام نشود (runaway). ۱۸ امن است.')


def probe_duration(path):
    ff = shutil.which('ffprobe')
    if not ff:
        return None
    r = subprocess.run([ff, '-v', 'error', '-show_entries', 'format=duration',
                        '-of', 'json', path], capture_output=True, text=True)
    try:
        return float(json.loads(r.stdout)['format']['duration'])
    except Exception:
        return None


for f in ('ref_voice.wav', 'ref_voice.mp3', 'ref_voice.m4a'):
    if os.path.exists(f):
        os.remove(f)

print('📁 لطفاً فایل صوتی صدای مرجع را آپلود کنید (۳ تا ۵ ثانیه، یک گوینده، بدون موسیقی):')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('❌ فایلی آپلود نشد!')

src = list(uploaded.keys())[0]
dur = probe_duration(src)

if dur is not None:
    print(f'⏱️ طول فایل آپلودی: {dur:.2f} ثانیه')
    if dur < 3:
        print('⚠️ کوتاه‌تر از ۳ ثانیه — کلونینگ ضعیف می‌شود.')
    if dur > Voice_Prompt_Seconds:
        print(f'✂️ به {Voice_Prompt_Seconds} ثانیه بریده می‌شود.')

REF_AUDIO_PATH = 'ref_voice.wav'
cmd = ['ffmpeg', '-y', '-loglevel', 'error', '-i', src, '-ar', str(SAMPLE_RATE), '-ac', '1']
if dur is not None and dur > Voice_Prompt_Seconds:
    cmd += ['-t', str(int(Voice_Prompt_Seconds))]
cmd.append(REF_AUDIO_PATH)
subprocess.run(cmd, check=True)

ref_dur = probe_duration(REF_AUDIO_PATH)

print(f'✅ صدای مرجع ذخیره شد: {REF_AUDIO_PATH} ({ref_dur:.2f} ثانیه، mono {SAMPLE_RATE}Hz)')

# استخراج و کش voice state — get_state_for_audio_prompt کند است، یک‌بار کافی است
print('⏳ استخراج voice state ...')
try:
    voice_state = tts_model.get_state_for_audio_prompt(REF_AUDIO_PATH, truncate=True)
except TypeError:
    voice_state = tts_model.get_state_for_audio_prompt(REF_AUDIO_PATH)

print('✅ voice state آماده شد.')
print(f'🎛️ حداکثر توکن فونم در هر تکه: {Max_Phoneme_Tokens} | سکوت بین تکه‌ها: {Pause_Between_Segments_ms}ms')


In [ ]:
#@title ▶️ شروع فرآیند: خواندن فایل با صدای کلون‌شده

import os, re, json, random, shutil, subprocess, traceback
import numpy as np
import torch
import scipy.io.wavfile as wavfile
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, HTML, Audio

YOUTUBE_CHANNEL_URL = "https://www.youtube.com/@aigolden?sub_confirmation=1"

SUBTITLE_EXTS = ('.srt', '.vtt', '.sub', '.ass')
SILENCE_MS = int(Pause_Between_Segments_ms)
OUTPUT_FILE = 'cloned_voice_docx_reading.mp3'
SEG_DIR = 'voice_clone_segments'
FAIL_FILE = 'failed_paragraphs.txt'

# Training utterances averaged ~11 tokens; 7 renders cleanly, 3 makes the model
# continue the voice prompt instead of the text.
MIN_PHONEME_TOKENS = 6


def probe_duration(path):
    """مدت فایل صوتی به ثانیه (None اگر ffprobe نبود)."""
    ff = shutil.which('ffprobe')
    if not ff:
        return None
    r = subprocess.run([ff, '-v', 'error', '-show_entries', 'format=duration',
                        '-of', 'json', path], capture_output=True, text=True)
    try:
        return float(json.loads(r.stdout)['format']['duration'])
    except Exception:
        return None


# ───────────────────────── استخراج متن
def _clean_subtitle(raw):
    out = []
    for block in re.split(r'\n\s*\n', raw):
        lines = []
        for line in block.splitlines():
            s = line.strip()
            if not s:
                continue
            if s.isdigit():
                continue
            if '-->' in s:
                continue
            if re.match(r'^(WEBVTT|NOTE\b|STYLE\b|REGION\b)', s, re.I):
                continue
            if re.match(r'^\d{1,2}:\d{2}:\d{2}[.,]\d{3}\s*$', s):
                continue
            lines.append(s)
        if lines:
            out.append(' '.join(lines))
    return out


def _iter_docx_blocks(doc):
    from docx.document import Document as _Doc
    from docx.oxml.table import CT_Tbl
    from docx.oxml.text.paragraph import CT_P
    from docx.table import Table, _Cell
    from docx.text.paragraph import Paragraph

    def walk(parent):
        if isinstance(parent, _Doc):
            elm = parent.element.body
        elif isinstance(parent, _Cell):
            elm = parent._tc
        else:
            return
        for child in elm.iterchildren():
            if isinstance(child, CT_P):
                yield Paragraph(child, parent)
            elif isinstance(child, CT_Tbl):
                yield Table(child, parent)

    yield from walk(doc)


def extract_paragraphs(path):
    ext = path.rsplit('.', 1)[-1].lower()

    if ext == 'docx':
        import docx
        from docx.table import Table

        doc = docx.Document(path)
        out = []

        for block in _iter_docx_blocks(doc):
            if isinstance(block, Table):
                for row in block.rows:
                    cells, seen = [], set()
                    for cell in row.cells:
                        if id(cell._tc) in seen:
                            continue
                        seen.add(id(cell._tc))
                        t = ' '.join(p.text.strip() for p in cell.paragraphs if p.text.strip())
                        if t:
                            cells.append(t)
                    if cells:
                        out.append(' ، '.join(cells))
            else:
                t = block.text.strip()
                if t:
                    out.append(t)
        return out

    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        raw = f.read()

    if f'.{ext}' in SUBTITLE_EXTS:
        return _clean_subtitle(raw)

    out = []
    for block in re.split(r'\n\s*\n', raw):
        lines = [l.strip() for l in block.splitlines() if l.strip()]
        if lines:
            out.append(' '.join(lines))
    return out


def merge_short(paras, min_len=25):
    merged = []
    for p in paras:
        if merged and len(p) < min_len:
            merged[-1] = merged[-1].rstrip() + ' ' + p
        else:
            merged.append(p)
    return merged


# ───────────────────────── تکه‌بندی
# G2P نشانه‌گذاری را دور می‌ریزد، پس باید *قبل* از فونم‌سازی تقسیم کرد؛
# وگرنه هر تکه‌بندی فقط روی تعداد توکن می‌برد و وسط عبارت می‌شکند.
def split_units(text):
    """جمله و عبارت را جدا می‌کند تا مکث‌ها سرِ جای خودشان بیفتند."""
    units = []
    for sent in re.split(r'(?<=[.!؟])\s+', text):
        for clause in re.split(r'(?<=[،؛:])\s+', sent):
            c = clause.strip()
            if c:
                units.append(c)
    return units or [text.strip()]


def build_chunks(paragraph):
    """پاراگراف فارسی → لیست (متن، فونم) با طول کنترل‌شده."""
    chunks, cur_text, cur_ph, cur_n = [], [], [], 0

    for unit in split_units(paragraph):
        try:
            ph = phonemise(unit)
        except Exception as e:
            print(f'   ⚠️ فونم‌سازی ناموفق: «{unit[:40]}» — {type(e).__name__}: {e}')
            continue
        if not ph:
            continue
        n = len(ph.split())

        if cur_n and cur_n + n > Max_Phoneme_Tokens:
            chunks.append((' '.join(cur_text), ' '.join(cur_ph)))
            cur_text, cur_ph, cur_n = [], [], 0

        cur_text.append(unit)
        cur_ph.append(ph)
        cur_n += n

    if cur_text:
        chunks.append((' '.join(cur_text), ' '.join(cur_ph)))

    # تکه‌ی آخرِ خیلی کوتاه را به قبلی بچسبان (۳ توکن = مدل پرامپت را ادامه می‌دهد)
    if len(chunks) >= 2 and len(chunks[-1][1].split()) < MIN_PHONEME_TOKENS:
        t, p = chunks.pop()
        pt, pp = chunks.pop()
        chunks.append((pt + ' ' + t, pp + ' ' + p))

    return chunks


# ───────────────────────── تولید صدا
def synth(phonemes, out_path):
    """فونم → wav. تشخیص runaway و پرتاب استثنا برای retry."""
    audio = tts_model.generate_audio(
        voice_state, phonemes,
        frames_after_eos=int(Frames_After_Eos),
    )
    a = audio.detach().cpu().numpy() if isinstance(audio, torch.Tensor) else np.asarray(audio)
    a = np.asarray(a, dtype=np.float32).ravel()

    dur = len(a) / SAMPLE_RATE
    expected = len(phonemes.split()) / 3.0 + 2.0

    if dur > expected * 2.2:
        raise RuntimeError(
            f'runaway: {dur:.1f}s تولید شد در حالی که انتظار ~{expected:.1f}s بود'
        )
    if dur < 0.25:
        raise RuntimeError(f'خروجی تقریباً خالی است ({dur:.2f}s)')

    a = np.clip(a, -1.0, 1.0)
    wavfile.write(out_path, SAMPLE_RATE, (a * 32767).astype(np.int16))
    return dur


def concat_segments(seg_paths, out_path, gap_ms=SILENCE_MS, work_dir=None):
    if not seg_paths:
        raise RuntimeError('هیچ سگمنتی تولید نشد!')

    work_dir = work_dir or SEG_DIR
    os.makedirs(work_dir, exist_ok=True)

    silence = os.path.join(work_dir, '_silence.wav')
    ff = shutil.which('ffmpeg')

    if ff:
        subprocess.run(
            [ff, '-y', '-loglevel', 'error', '-f', 'lavfi',
             '-i', f'anullsrc=r={SAMPLE_RATE}:cl=mono', '-t', str(gap_ms / 1000), silence],
            check=True,
        )
        lst = os.path.join(work_dir, '_list.txt')
        with open(lst, 'w', encoding='utf-8') as f:
            for i, p in enumerate(seg_paths):
                f.write(f"file '{os.path.abspath(p)}'\n")
                if i != len(seg_paths) - 1 and gap_ms > 0:
                    f.write(f"file '{os.path.abspath(silence)}'\n")

        r = subprocess.run(
            [ff, '-y', '-loglevel', 'error', '-f', 'concat', '-safe', '0',
             '-i', lst, '-c:a', 'libmp3lame', '-b:a', '192k', out_path],
            capture_output=True, text=True,
        )
        if r.returncode == 0 and os.path.exists(out_path):
            return out_path
        print(f'⚠️ ffmpeg ناموفق بود ({r.stderr.strip()[:200]}) — از pydub استفاده می‌کنم.')

    from pydub import AudioSegment
    final = AudioSegment.empty()
    for i, p in enumerate(seg_paths):
        final += AudioSegment.from_file(p)
        if i != len(seg_paths) - 1 and gap_ms > 0:
            final += AudioSegment.silent(duration=gap_ms)
    final.export(out_path, format='mp3', bitrate='192k')
    return out_path


# ───────────────────────── اجرا
if 'SELECTED_DOC_PATH' not in globals() or not SELECTED_DOC_PATH:
    raise RuntimeError('❌ ابتدا در سلول «انتخاب فایل متنی» یک فایل انتخاب کنید!')
if 'voice_state' not in globals():
    raise RuntimeError('❌ ابتدا سلول «آپلود صدای مرجع» را اجرا کنید!')

print(f'📄 خواندن متن از: {SELECTED_DOC_PATH}')
PARAGRAPHS = merge_short(extract_paragraphs(SELECTED_DOC_PATH))

if not PARAGRAPHS:
    raise RuntimeError('❌ متنی در فایل یافت نشد!')

print(f'✅ {len(PARAGRAPHS)} پاراگراف ({sum(len(p) for p in PARAGRAPHS)} کاراکتر).')
print('⏳ فونم‌سازی و تکه‌بندی ...')

# ساخت صف کار: هر آیتم یک تکه‌ی فونم با برچسب پاراگراف
JOB_QUEUE = []
for pi, para in enumerate(PARAGRAPHS):
    for ci, (txt, ph) in enumerate(build_chunks(para)):
        JOB_QUEUE.append({'para': pi, 'chunk': ci, 'text': txt, 'ph': ph})

if not JOB_QUEUE:
    raise RuntimeError('❌ هیچ فونم معتبری تولید نشد — متن احتمالاً فارسی نیست.')

print(f'✅ {len(JOB_QUEUE)} تکه برای سنتز آماده شد.')
print()
print('🔍 پیش‌نمایش سه تکه‌ی اول:')
for j in JOB_QUEUE[:3]:
    print(f'   «{j["text"][:50]}»')
    print(f'   → {j["ph"][:70]}')

os.makedirs(SEG_DIR, exist_ok=True)
for f in os.listdir(SEG_DIR):
    if f.startswith('seg_'):
        os.remove(os.path.join(SEG_DIR, f))

seg_paths, failures = [], []

print()
for i, job in enumerate(JOB_QUEUE):
    seg = os.path.join(SEG_DIR, f'seg_{i+1:05d}.wav')
    preview = job['text'][:40]

    for attempt in range(1, 5):          # runaway تصادفی است؛ ۴ تلاش
        try:
            dur = synth(job['ph'], seg)
            seg_paths.append(seg)
            print(f'   ✅ [{i+1}/{len(JOB_QUEUE)}] {dur:.1f}s — {preview}...')
            break
        except Exception as e:
            if attempt == 4:
                failures.append((job['para'] + 1, job['text'], repr(e)))
                print(f'   ❌ [{i+1}/{len(JOB_QUEUE)}] پس از ۴ تلاش رد شد: {e}')
                traceback.print_exc(limit=1)
            else:
                print(f'   ↻ [{i+1}] تلاش {attempt} ناموفق ({type(e).__name__}) — تلاش مجدد...')

    if GPU_OK and i % 20 == 19:
        torch.cuda.empty_cache()

print()
print(f'🎬 ساخت تراک نهایی از {len(seg_paths)} تکه ...')
output_file = concat_segments(seg_paths, OUTPUT_FILE)

if failures:
    with open(FAIL_FILE, 'w', encoding='utf-8') as f:
        for idx, txt, err in failures:
            f.write(f'[پاراگراف {idx}] {err}\n{txt}\n\n')
    print(f'⚠️ {len(failures)} تکه خوانده نشد — جزئیات در {FAIL_FILE}')
    for idx, txt, err in failures[:5]:
        print(f'   • [پاراگراف {idx}] {txt[:60]}... → {err}')

size_mb = os.path.getsize(output_file) / 1e6
total_dur = probe_duration(output_file)
print(f'🎉 تراک نهایی: {output_file} ({size_mb:.1f} MB، {total_dur:.1f} ثانیه)')


# ───────────────────────── پخش و دانلود
display(Audio(output_file, autoplay=True))

btn_download = widgets.Button(description='⬇️ دانلود تراک نهایی', button_style='success')
btn_download.on_click(lambda b: files.download(output_file))
display(btn_download)

if failures:
    btn_fail = widgets.Button(description='⬇️ دانلود لیست تکه‌های ناموفق', button_style='warning')
    btn_fail.on_click(lambda b: files.download(FAIL_FILE))
    display(btn_fail)

display(HTML(
    f'<a href="{YOUTUBE_CHANNEL_URL}" target="_blank" '
    f'style="display:inline-block;margin-top:10px;padding:10px 22px;background:#FF0000;'
    f'color:#fff;border-radius:6px;text-decoration:none;font-family:sans-serif;font-weight:bold;">'
    f'🔔 ما را در یوتیوب دنبال کنید</a>'
))


In [ ]:
#@title ✏️ اصلاح دستی تلفظ (جایگزینی در فایل و در حافظه)

import os
import ipywidgets as widgets
from IPython.display import display

if 'SELECTED_DOC_PATH' not in globals() or not SELECTED_DOC_PATH or not os.path.exists(SELECTED_DOC_PATH):
    raise RuntimeError('❌ ابتدا سلول «انتخاب فایل متنی» را اجرا کنید!')

CURRENT_FILE = SELECTED_DOC_PATH
FILE_EXT = CURRENT_FILE.rsplit('.', 1)[-1].lower()


def _replace_in_paragraph(p, w, c):
    full = p.text
    n = full.count(w)
    if not n:
        return 0

    hit = False
    for run in p.runs:
        if w in run.text:
            run.text = run.text.replace(w, c)
            hit = True

    if not hit:
        for run in list(p.runs)[1:]:
            run.text = ''
        if p.runs:
            p.runs[0].text = full.replace(w, c)
    return n


def replace_in_file(w, c):
    total = 0

    if FILE_EXT == 'docx':
        import docx
        doc = docx.Document(CURRENT_FILE)
        for p in doc.paragraphs:
            total += _replace_in_paragraph(p, w, c)
        for table in doc.tables:
            for row in table.rows:
                for cell in row.cells:
                    for p in cell.paragraphs:
                        total += _replace_in_paragraph(p, w, c)
        if total:
            doc.save(CURRENT_FILE)
        return total

    with open(CURRENT_FILE, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.read().splitlines()

    new_lines = []
    for line in lines:
        total += line.count(w)
        new_lines.append(line.replace(w, c))

    if total:
        with open(CURRENT_FILE, 'w', encoding='utf-8') as f:
            f.write('\n'.join(new_lines))
    return total


wrong_word = widgets.Text(description='کلمه اشتباه:', placeholder='مثلاً: نکته',
                          layout=widgets.Layout(width='460px'))
correct_word = widgets.Text(description='عبارت درست:', placeholder='مثلاً: نُکته',
                            layout=widgets.Layout(width='460px'))
btn_replace = widgets.Button(description='🔁 جایگزینی', button_style='warning')
btn_preview = widgets.Button(description='🔍 پیش‌نمایش فونم', button_style='info')
log_output = widgets.Output()

display(widgets.HTML(f'<b>📄 فایل فعلی:</b> {CURRENT_FILE}'),
        wrong_word, correct_word, widgets.HBox([btn_replace, btn_preview]), log_output)


def on_replace_clicked(b):
    with log_output:
        log_output.clear_output()

        w = wrong_word.value.strip()
        c = correct_word.value.strip()

        if not w or not c:
            print('⚠️ هر دو فیلد را پر کنید.')
            return
        if w == c:
            print('⚠️ عبارت اشتباه و درست یکسان‌اند.')
            return

        n_file = replace_in_file(w, c)

        n_mem = 0
        if 'PARAGRAPHS' in globals() and PARAGRAPHS:
            for i, p in enumerate(PARAGRAPHS):
                if w in p:
                    n_mem += p.count(w)
                    PARAGRAPHS[i] = p.replace(w, c)

        if n_file == 0 and n_mem == 0:
            print(f'ℹ️ عبارت «{w}» پیدا نشد.')
            return

        print(f'✅ {n_file} مورد در فایل و {n_mem} مورد در حافظه جایگزین شد.')
        print(f'🔍 فونم جدید: {phonemise(c)}')
        wrong_word.value = ''
        correct_word.value = ''


def on_preview_clicked(b):
    with log_output:
        log_output.clear_output()
        t = wrong_word.value.strip()
        if not t:
            print('⚠️ متنی برای پیش‌نمایش بنویسید.')
            return
        print(f'«{t}»')
        print(f'→ {phonemise(t)}')
        print()
        print('اگر فونم خالی است، G2P نتوانسته بخواند — شکل دیگری بنویسید.')


btn_replace.on_click(on_replace_clicked)
btn_preview.on_click(on_preview_clicked)

print('راهنما: کلمه‌ی اشتباه‌خوانده‌شده را در فیلد اول و شکل درست (با اعراب/نیم‌فاصله) را در فیلد دوم بنویس. '
      'با «🔍 پیش‌نمایش فونم» ببین G2P چه خوانشی می‌دهد، بعد «🔁 جایگزینی» را بزن. '
      'در پایان سلول «▶️ شروع فرآیند» را دوباره اجرا کن.')


In [ ]:
#@title 🧪 (اختیاری) تست دستی بدون فایل — وارد کردن متن مستقیم

# این سلول از توابع سلول «▶️ شروع فرآیند» استفاده می‌کند،
# پس یک بار آن سلول را اجرا کنید (حتی اگر فایل ندارید).

import os
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, HTML, Audio

Manual_Text = "" #@param {type:"string"}
YOUTUBE_CHANNEL_URL = "https://www.youtube.com/@aigolden?sub_confirmation=1"

if 'synth' not in globals() or 'build_chunks' not in globals():
    raise RuntimeError('❌ ابتدا یک بار سلول «▶️ شروع فرآیند» را اجرا کنید تا توابع آماده شوند.')
if 'voice_state' not in globals():
    raise RuntimeError('❌ ابتدا سلول «آپلود صدای مرجع» را اجرا کنید!')

paragraphs = [l.strip() for l in Manual_Text.splitlines() if l.strip()]
if not paragraphs:
    raise RuntimeError('❌ متنی وارد نشده است!')

MANUAL_SEG_DIR = 'voice_clone_segments_manual'
os.makedirs(MANUAL_SEG_DIR, exist_ok=True)

jobs = []
for para in paragraphs:
    jobs.extend(build_chunks(para))

if not jobs:
    raise RuntimeError('❌ فونمی تولید نشد — متن فارسی است؟')

print(f'🔍 {len(jobs)} تکه:')
for txt, ph in jobs:
    print(f'   «{txt[:45]}»  →  {ph[:60]}')

seg_paths, failures = [], []
for i, (txt, ph) in enumerate(jobs):
    seg = os.path.join(MANUAL_SEG_DIR, f'seg_{i+1:05d}.wav')
    for attempt in range(1, 5):
        try:
            dur = synth(ph, seg)
            seg_paths.append(seg)
            print(f'   ✅ [{i+1}/{len(jobs)}] {dur:.1f}s — {txt[:40]}...')
            break
        except Exception as e:
            if attempt == 4:
                failures.append((i + 1, txt, repr(e)))
                print(f'   ❌ [{i+1}] رد شد: {e}')
            else:
                print(f'   ↻ [{i+1}] تلاش {attempt} ناموفق — تلاش مجدد...')

output_file = concat_segments(seg_paths, 'cloned_voice_manual_test.mp3', work_dir=MANUAL_SEG_DIR)

print(f'🎉 تراک آزمایشی آماده شد: {output_file}')
if failures:
    print(f'⚠️ {len(failures)} تکه رد شد.')
display(Audio(output_file, autoplay=True))

btn_download = widgets.Button(description='⬇️ دانلود تراک آزمایشی', button_style='success')
btn_download.on_click(lambda b: files.download(output_file))
display(btn_download)

display(HTML(
    f'<a href="{YOUTUBE_CHANNEL_URL}" target="_blank" '
    f'style="display:inline-block;margin-top:10px;padding:10px 22px;'
    f'background:#FF0000;color:#fff;border-radius:6px;'
    f'text-decoration:none;font-family:sans-serif;font-weight:bold;">'
    f'🔔 ما را در یوتیوب دنبال کنید</a>'
))


In [ ]:
#@title 🧹 پاکسازی فایل‌های موقت

import os, glob, shutil

#@markdown فایل‌های mp3 و پوشه‌ی Drive دست‌نخورده می‌مانند.
Wipe_Everything = False #@param {type:"boolean"}

KEEP_SUFFIXES = ('.mp3',)

removed = []

for pattern in ('/content/voice_clone_segments*', '/content/ref_voice.*',
                '/content/_silence.wav', '/content/_list.txt',
                '/content/failed_paragraphs.txt'):
    for p in glob.glob(pattern):
        if os.path.isdir(p):
            shutil.rmtree(p, ignore_errors=True)
        else:
            try:
                os.remove(p)
            except OSError:
                continue
        removed.append(p)

if Wipe_Everything:
    for entry in os.listdir('/content'):
        if entry == 'drive':
            continue
        if entry.lower().endswith(KEEP_SUFFIXES):
            continue
        full = os.path.join('/content', entry)
        try:
            shutil.rmtree(full) if os.path.isdir(full) else os.remove(full)
            removed.append(full)
        except OSError:
            pass

print('🧹 حذف شد:' if removed else 'ℹ️ چیزی برای حذف نبود.')
for p in removed:
    print('  •', p)


## ⚠️ راهنمای این نسخه (pocket-tts-farsi-v2)

### چرا این مدل؟

`mehdi-hf/pocket-tts-farsi-v2` — ۱۰۹.۵ میلیون پارامتر، آموزش روی **۹۷۳ ساعت** گفتار فارسی از **۲۹۷۸ گوینده**.
اعداد روی ۳۰۰ نمونه‌ی Common Voice که مدل ندیده:

| معیار | v1 | **v2** |
|---|---|---|
| WER میانگین | 2.048 | **0.582** |
| شباهت گوینده | 0.764 | **0.859** |
| UTMOS (طبیعی‌بودن) | 2.826 | **3.11** |
| runaway | ۲۵ از ۳۰۰ | **۰–۱ از ۳۰۰** |

### سه چیزی که باید بدانید

**۱. صدای مرجع حداکثر ۵ ثانیه.** مهم‌ترین قاعده. مدل روی پرامپت ≤۵ ثانیه آموزش دیده؛ بلندتر باعث می‌شود مدل جمله‌ی خودِ پرامپت را ادامه دهد نه متن شما. اندازه‌گیری رسمی: ۶.۱ ثانیه → ۰ از ۴ بار؛ همان کلیپ در ۵ ثانیه → ۴ از ۴ بار. سلول آپلود به‌طور خودکار می‌برَد.

**۲. مدل خط فارسی نمی‌خواند.** ورودی‌اش فونم رومانیزه است. متن فارسی مستقیم = **سکوت مطلق، بدون هیچ خطا**. برای همین سلول «🧠 لود مدل» یک موتور G2P (`Homo-GE2PE-Persian-HF`) هم لود می‌کند و هر پاراگراف قبل از سنتز فونم می‌شود.

**۳. حداکثر ۱۸ توکن فونم در هر تکه.** بالای ۲۰ توکن تولید ممکن است هرگز تمام نشود. تکه‌ی خیلی کوتاه (زیر ۶ توکن) هم باعث می‌شود مدل پرامپت را ادامه دهد.

### چرا تقسیم روی متن فارسی و نه فونم؟

G2P همه‌ی نشانه‌گذاری را دور می‌ریزد. اگر اول فونم کنید و بعد تکه‌بندی، هیچ مرز جمله‌ای باقی نمی‌ماند و تکه‌ها وسط عبارت می‌شکنند. این نسخه اول جمله و عبارت را جدا می‌کند (`.!؟` و `،؛:`)، هر واحد را جدا فونم می‌کند، بعد واحدهای کوچک را تا سقف ۱۸ توکن به هم می‌چسباند.

### runaway چیست؟

تولیدی که هرگز `end-of-speech` نمی‌دهد و تا سقف طول تکرار می‌کند — `fanAvari` می‌شود `fanAvariiiiii…`. تصادفی است و تلاش دوم معمولاً تمام می‌شود. کد مدت خروجی را با `توکن ÷ ۳ + ۲` ثانیه مقایسه می‌کند و اگر بیش از ۲.۲ برابر بود، دوباره تولید می‌کند (تا ۴ تلاش).

### محدودیت‌های شناخته‌شده

- **کلمه‌ی اول هر تکه** ضعیف‌ترین نقطه است: حدود ۴۰ بار از ۵۰ درست، در برابر ۴۶ از ۵۰ برای همان کلمه یک پله جلوتر. حرف انسدادی آغازین می‌لرزد (`کریس` → گاهی `پریس`).
- **هیچ نشانه‌گذاری‌ای در واژگان مدل نیست.** نمی‌توانید با کاما مکث بخواهید؛ جمله‌بندی کاملاً کار خود مدل است. تنها اهرم شما مکث بین تکه‌هاست (تنظیم `Pause_Between_Segments_ms`).
- **WER مطلق بالا است** چون Whisper خودش روی Common Voice خطای ۰.۲۷۳ دارد. این اعداد برای مقایسه‌ی مدل‌ها هستند، نه نمره‌ی کیفیت مطلق.
- **مجوز CC-BY-NC-4.0** — غیرتجاری.

### اگر مجوز تجاری می‌خواهید

`mehdi-hf/chatterbox-farsi` (مجوز MIT، خط فارسی را مستقیم می‌خواند، WER ۱۲.۵٪ زن / ۱۶.۹٪ مرد). نیازی به G2P ندارد ولی ۵ برابر بزرگ‌تر است و روی CPU کند.

### تفاوت با نسخه‌ی OmniVoice

| | OmniVoice | pocket-tts-farsi-v2 |
|---|---|---|
| اندازه | ۳.۲۷ GB | ~۴۴۰ MB |
| ورودی | خط فارسی مستقیم | فونم (نیاز به G2P) |
| WER فارسی | منتشر نشده | **0.582** |
| سرعت CPU | کند | **~۶× سریع‌تر از زمان واقعی** |
| مجوز | — | CC-BY-NC-4.0 |

OmniVoice روی GPU خام سریع‌تر است (RTF ~۰.۲۷) و ۶۰۰+ زبان دارد، ولی برای فارسی عدد منتشرشده ندارد. این مدل برای فارسی عدد دارد.
